
## Exercício 2 – Implementação manual do pipeline de processamento de texto

### Contexto:
O pipeline de processamento de texto é construído passo a passo, com o objetivo de compreender cada
transformação aplicada ao texto bruto.
Na próxima aulas, esse mesmo pipeline será implementado utilizando bibliotecas especializadas de PLN, como NLTK
e spaCy. Antes disso, é fundamental consolidar a compreensão do pipeline implementado manualmente, sem o
auxílio de bibliotecas linguísticas, de modo a evidenciar o papel de cada etapa como parte de um fluxo de
engenharia de software.

**Tarefa**:
Implemente manualmente, um pipeline de processamento de texto que realize as seguintes etapas:
1. Limpeza do texto
- Remoção de URLs
- Remoção de marcações HTML
- Remoção de emojis e caracteres especiais irrelevantes
2. Normalização textual
- Conversão para letras minúsculas
- Remoção de acentuação, exceto para a palavra: é
3. Tokenização
- Segmentação do texto em palavras
4. Remoção seletiva de stopwords
- Remoção apenas de palavras funcionais (artigos, preposições e conjunções)
- Preservação de palavras semanticamente relevantes, como adjetivos e verbos
5. Redução morfológica usando lema
6. Redução morfológica usando stemming

### Texto base para teste:
```
"O produto (http://exemplo.com) é bom ou ruim, mas não é <p title='Destaque'>perfeito</p> e tem defeitos" 
```

## Pipeline

### 0. Entrada de texto

In [8]:

texto_bruto = "O produto (http://exemplo.com) é bom ou ruim, mas não é <p title='Destaque'>perfeito</p> e tem defeitos"
print("Texto bruto:", texto_bruto)


Texto bruto: O produto (http://exemplo.com) é bom ou ruim, mas não é <p title='Destaque'>perfeito</p> e tem defeitos


### 1. Limpeza do texto
- Remoção de URLs
- Remoção de marcações HTML
- Remoção de emojis e caracteres especiais irrelevantes

In [9]:

import re

def higienizar(texto: str) -> str:
    texto = re.sub(r"http\S+|www\.\S+", "", texto)
    texto = re.sub(r"<[^>]+>", " ", texto)
    texto = re.sub(r"[^\w\s]", " ", texto, flags=re.UNICODE)
    return texto

texto_limpo = higienizar(texto_bruto)
print("Após higienização:", texto_limpo)


Após higienização: O produto   é bom ou ruim  mas não é  perfeito  e tem defeitos


### 2. Normalização textual
- Conversão para letras minúsculas
- Remoção de acentuação, exceto para a palavra: é

In [10]:

import unicodedata

PALAVRAS_PRESERVADAS = {"é"}

def normalizar(texto: str, remover_acentos: bool = True) -> str:
    texto = texto.lower()
    tokens = texto.split()

    if remover_acentos:
        tokens_normalizados = []
        for token in tokens:
            if token in PALAVRAS_PRESERVADAS:
                tokens_normalizados.append(token)
            else:
                token = unicodedata.normalize("NFD", token)
                token = "".join(c for c in token if not unicodedata.combining(c))
                tokens_normalizados.append(token)

        texto = " ".join(tokens_normalizados)

    texto = re.sub(r"\s+", " ", texto).strip()
    return texto

texto_normalizado = normalizar(texto_limpo, remover_acentos=True)
print("Após normalização:", texto_normalizado)


Após normalização: o produto é bom ou ruim mas nao é perfeito e tem defeitos


### 3. Tokenização
- Segmentação do texto em palavras 

In [11]:

from typing import List

def tokenizar(texto: str) -> List[str]:
    return texto.split()

tokens = tokenizar(texto_normalizado)
print("Tokens:", tokens)


Tokens: ['o', 'produto', 'é', 'bom', 'ou', 'ruim', 'mas', 'nao', 'é', 'perfeito', 'e', 'tem', 'defeitos']


### 4. Remoção seletiva de stopwords
- Remoção apenas de palavras funcionais (artigos, preposições e conjunções)
- Preservação de palavras semanticamente relevantes, como adjetivos e verbos

In [12]:

STOPWORDS_MIN = {
    "o", "a", "os", "as",
    "de", "do", "da", "dos", "das",
    "e", "em", "no", "na", "nos", "nas",
    "com", "por", "para",
    "um", "uma", "uns", "umas",
    "ou", "mas"
}

def remover_stopwords(tokens: List[str], manter: set = None) -> List[str]:
    manter = manter or set()
    saida = []

    for t in tokens:
        if t in manter:
            saida.append(t)
        elif t not in STOPWORDS_MIN:
            saida.append(t)

    return saida

tokens_filtrados = remover_stopwords(tokens, manter={"nao"})
print("Sem stopwords:", tokens_filtrados)


Sem stopwords: ['produto', 'é', 'bom', 'ruim', 'nao', 'é', 'perfeito', 'tem', 'defeitos']


### 5. Redução morfológica usando Lematização:

In [13]:

LEMA_MANUAL = {
    "produto": "produto",
    "é": "ser",
    "tem": "ter",
    "defeitos": "defeito",
    "bom": "bom",
    "ruim": "ruim",
    "perfeito": "perfeito",
    "nao": "nao"
}

def lematizar(tokens: List[str]) -> List[str]:
    return [LEMA_MANUAL.get(t, t) for t in tokens]

tokens_reduzidos = lematizar(tokens_filtrados)
print("Redução morfológica (Lema):", tokens_reduzidos)


Redução morfológica (Lema): ['produto', 'ser', 'bom', 'ruim', 'nao', 'ser', 'perfeito', 'ter', 'defeito']


### 6. Redução morfológica usando Stemming

In [14]:

STEM_MANUAL = {
    "produto": "produt",
    "ser": "ser",
    "bom": "bom",
    "ruim": "ruim",
    "nao": "nao",
    "perfeito": "perfeit",
    "ter": "ter",
    "defeito": "defeit"
}

def stemming(tokens: List[str]) -> List[str]:
    return [STEM_MANUAL.get(t, t) for t in tokens]

tokens_stemizados = stemming(tokens_reduzidos)
print("Stemming:", tokens_stemizados)


Stemming: ['produt', 'ser', 'bom', 'ruim', 'nao', 'ser', 'perfeit', 'ter', 'defeit']
